In [2]:
import os
import numpy as np
import pandas as pd
import diptest

from sklearn.mixture import GaussianMixture

INPUT_CSV = r"C:\Users\ishin\OneDrive\Desktop\ish\gbm_thickness_summary.csv"
OUTPUT_CSV = r"C:\Users\ishin\OneDrive\Desktop\ish\gbm_multimodality_summary.csv"

if not os.path.exists(INPUT_CSV):
    raise FileNotFoundError(
        f"Input file not found:\n{INPUT_CSV}"
    )
df = pd.read_csv(INPUT_CSV)

print("GBM MULTIMODALITY ANALYSIS")
print("=" * 60)
print(f"Loaded {len(df)} membrane components")
print(f"Patients : {df['patient_id'].nunique()}")

results = []

def fit_gmm_models(values):
    X = values.reshape(-1, 1)
    models = {}
    bic = {}
    aic = {}

    for n in [1, 2, 3]:
        model = GaussianMixture(
            n_components=n,
            random_state=42,
            n_init=20
        )
        model.fit(X)
        models[n] = model
        bic[n] = model.bic(X)
        aic[n] = model.aic(X)

    return models, bic, aic

#Analyze per patient
for patient_id, group in df.groupby("patient_id"):

    print("\n" + "-" * 80)
    print(f"Patient : {patient_id}")

    thickness = (
        group["median_thickness_nm"]
        .dropna()
        .to_numpy()
    )
    n_samples = len(thickness)
    print(f"Membrane components : {n_samples}")

    # Too few samples
    if n_samples < 5:
        print("Skipped (too few membrane components)")
        continue

   
#Hartigan Dip Test 
    dip_statistic, dip_pvalue = diptest.diptest(thickness)
    dip_result = (
        "Multimodal"
        if dip_pvalue < 0.05
        else "Unimodal"
    )

#Gaussian model
    models, bic, aic = fit_gmm_models(thickness)
    best_components = min(
        bic,
        key=bic.get
    )
    best_model = models[best_components]

   #peak positions and weights
    peaks = np.sort(
        best_model.means_.flatten()
    )
    weights = best_model.weights_.flatten()

    order = np.argsort(
        best_model.means_.flatten()
    )
    weights = weights[order]

    
    peaks = np.pad(
        peaks,
        (0, 3 - len(peaks)),
        constant_values=np.nan
    )

    weights = np.pad(
        weights,
        (0, 3 - len(weights)),
        constant_values=np.nan
    )
   
    #peak separation
    if best_components >= 2:
        peak_distance = peaks[1] - peaks[0]
    else:
        peak_distance = np.nan

   
    print(f"Dip p-value      : {dip_pvalue:.5f}")
    print(f"Dip result       : {dip_result}")
    print(f"Best GMM         : {best_components} component(s)")
    print(f"BIC              : {bic[best_components]:.2f}")
    print(f"AIC              : {aic[best_components]:.2f}")
    print(f"Peak positions   : {np.round(peaks,2)}")
    print(f"Peak weights     : {np.round(weights,3)}")

   
    results.append({

        "Patient_ID": patient_id,
        "Number_of_membranes": n_samples,
        "Dip_statistic": dip_statistic,
        "Dip_p_value": dip_pvalue,
        "Dip_result": dip_result,
        "Best_GMM_components": best_components,

        "BIC": bic[best_components],
        "AIC": aic[best_components],

        "Peak_1_nm": peaks[0],
        "Peak_2_nm": peaks[1],
        "Peak_3_nm": peaks[2],

        "Weight_1": weights[0],
        "Weight_2": weights[1],
        "Weight_3": weights[2],

        "Peak_separation_nm": peak_distance
    })


summary_df = pd.DataFrame(results)
if summary_df.empty:
    print("\n No valid patients were analysed.")
    raise SystemExit

summary_df = summary_df.sort_values(
    by=[
        "Dip_p_value",
        "Best_GMM_components",
        "BIC"
    ],
    ascending=[
        True,
        False,
        True
    ]
).reset_index(drop=True)

summary_df.insert(
    0,
    "Rank",
    np.arange(1, len(summary_df) + 1)
)

#multimodality strength classification
def classify_strength(row):

    if (
        row["Dip_p_value"] < 0.01 and
        row["Best_GMM_components"] >= 3
    ):
        return "Very Strong"

    elif (
        row["Dip_p_value"] < 0.05 and
        row["Best_GMM_components"] >= 2
    ):
        return "Strong"

    elif row["Best_GMM_components"] >= 2:
        return "Moderate"

    else:
        return "Weak"

summary_df["Evidence"] = summary_df.apply(
    classify_strength,
    axis=1
)

summary_df.to_csv(
    OUTPUT_CSV,
    index=False
)

print("\n")
print("PATIENT MULTIMODALITY RANKING")
print("=" * 60)

display_columns = [

    "Rank",
    "Patient_ID",
    "Evidence",
    "Dip_p_value",
    "Best_GMM_components",
    "Peak_1_nm",
    "Peak_2_nm",
    "Peak_3_nm"
]

print(
    summary_df[
        display_columns
    ].to_string(index=False)
)

print("\n")
print("SUMMARY")
print("=" * 60)

print(f"Patients analysed        : {len(summary_df)}")

print(
    f"Multimodal patients      : "
    f"{(summary_df['Best_GMM_components'] > 1).sum()}"
)

print(
    f"Unimodal patients        : "
    f"{(summary_df['Best_GMM_components'] == 1).sum()}"
)

print("\nResults saved to:")
print(OUTPUT_CSV)

print("Analysis completed successfully")
print("=" * 60)

GBM MULTIMODALITY ANALYSIS
Loaded 394 membrane components
Patients : 11

--------------------------------------------------------------------------------
Patient : 01-24
Membrane components : 19
Dip p-value      : 0.86254
Dip result       : Unimodal
Best GMM         : 3 component(s)
BIC              : 204.34
AIC              : 196.78
Peak positions   : [170.91 372.42 480.62]
Peak weights     : [0.842 0.053 0.105]

--------------------------------------------------------------------------------
Patient : 02-24
Membrane components : 41
Dip p-value      : 0.99123
Dip result       : Unimodal
Best GMM         : 3 component(s)
BIC              : 477.14
AIC              : 463.44
Peak positions   : [ 241.72  785.13 1339.96]
Peak weights     : [0.951 0.024 0.024]

--------------------------------------------------------------------------------
Patient : 03-24
Membrane components : 22
Dip p-value      : 0.80725
Dip result       : Unimodal
Best GMM         : 2 component(s)
BIC              : 263.